## Prerequisites

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip drive/MyDrive/archive.zip

In [ ]:
!pip install monai

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image

from sklearn.model_selection import KFold
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### Dataset and Helper Functions

In [ ]:
def process_csv(file_path):
    df = pd.read_csv(file_path)
    images_dict = {}

    for i in range(len(df)):
        id_full = df.iloc[i]['ID']
        label_value = df.iloc[i]['Label']

        parts = id_full.split('_')
        hemorrhage_type = parts[-1]
        image_id = '_'.join(parts[:-1])

        if image_id not in images_dict:
            images_dict[image_id] = {}

        images_dict[image_id][hemorrhage_type] = label_value

    return images_dict

In [ ]:
class RSNAHemorrhageDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.images_dict = process_csv(csv_file)
        self.image_ids = list(self.images_dict.keys())
        self.label_columns = ['epidural', 'intraparenchymal', 'intraventricular',
                             'subarachnoid', 'subdural', 'any']

        print(f"Dataset initialized with {len(self)} images from {img_dir}")
        print(f"Categories: {self.label_columns}")

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_name = img_id + '_frame0.png'
        img_path = os.path.join(self.img_dir, img_name)

        data = {'image': img_path}

        if self.transform:
            data = self.transform(data)

        image = data['image']

        labels = []
        for label_col in self.label_columns:
            label_value = self.images_dict[img_id].get(label_col, 0)
            labels.append(label_value)

        label = torch.tensor(labels, dtype=torch.float32)

        return image, label

    def get_image_id(self, idx):
        return self.image_ids[idx]

### Pretrained used model

In [ ]:
import torchvision.models as models
import torch.nn as nn

In [ ]:
class IntracranialHemorrhageEfficientNet(nn.Module):
    NUM_CLASSES = 6

    def __init__(self, num_classes=NUM_CLASSES, pretrained=True, unfreeze_layers=-1, dropout_prob=0.3):
        super(IntracranialHemorrhageEfficientNet, self).__init__()
        self.efficientnet = models.efficientnet_b7(pretrained=pretrained)
        self.unfreeze_layers = unfreeze_layers

        # modify first conv layer for grayscale input
        original_input = self.efficientnet.features[0][0]
        self.efficientnet.features[0][0] = nn.Conv2d(
            in_channels=1,
            out_channels=original_input.out_channels,
            kernel_size=original_input.kernel_size,
            stride=original_input.stride,
            padding=original_input.padding,
            bias=False
        )

        # replace classifier with a dropout layer and a new linear layer
        num_features = self.efficientnet.classifier[1].in_features

        # without dropout
        # self.efficientnet.classifier = nn.Linear(num_features, num_classes)

        # for dropout
        self.efficientnet.classifier = nn.Sequential(
            nn.Dropout(p=dropout_prob), # Dropout layer added here
            nn.Linear(num_features, num_classes)
        )

        # apply selective freezing
        self.freeze_layers()

    def freeze_layers(self):
        if self.unfreeze_layers == -1:
            for param in self.efficientnet.parameters(): # unfreeze everything
                param.requires_grad = True
            return

        # first, freeze everything
        for param in self.efficientnet.parameters():
            param.requires_grad = False

        # get all feature blocks
        feature_blocks = list(self.efficientnet.features.children())
        total_blocks = len(feature_blocks)

        if self.unfreeze_layers != 0: # unfreeze last N blocks
            unfreeze_from = max(0, total_blocks - self.unfreeze_layers)
            for i in range(unfreeze_from, total_blocks):
                for param in feature_blocks[i].parameters():
                    param.requires_grad = True

            print(f"Unfrozen blocks: {unfreeze_from} to {total_blocks-1} (out of {total_blocks})")

        # always unfreeze classifier
        for param in self.efficientnet.classifier.parameters():
            param.requires_grad = True

    def forward(self, x):
        x = self.efficientnet(x)
        return x

### Training and Evaluation Functions

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device, epoch_num):
    model.train()

    running_loss = 0.0
    correct_samples = 0
    total_samples = 0

    all_predictions_for_metrics = []        # for precision, recall, f1, hamming score
    all_labels_for_metrics = []             # for precision, recall, f1, hamming score
    all_probabilities_for_metrics = []      # for AUC

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        probabilities = torch.sigmoid(outputs)
        predicted = (probabilities > 0.5).float()
        correct_samples += (predicted == labels).all(dim=1).sum().item()
        total_samples += labels.size(0)

        all_predictions_for_metrics.append(predicted.cpu().numpy())
        all_labels_for_metrics.append(labels.cpu().numpy())
        all_probabilities_for_metrics.append(probabilities.cpu().detach().numpy())

        # print progress every 10 batches (using user's accuracy method)
        if (batch_idx + 1) % 10 == 0:
            current_loss = running_loss / (batch_idx + 1)
            current_accuracy = correct_samples / total_samples if total_samples > 0 else 0.0

            print(f"Epoch {epoch_num}, Batch [{batch_idx+1}/{len(train_loader)}] - "
                  f"Loss: {current_loss:.4f} | Acc: {current_accuracy:.4f}")

    avg_loss = running_loss / len(train_loader)
    accuracy = correct_samples / total_samples if total_samples > 0 else 0.0

    # calculate other metrics using collected predictions/labels
    final_predictions_for_metrics = np.vstack(all_predictions_for_metrics)
    final_labels_for_metrics = np.vstack(all_labels_for_metrics)
    final_probabilities_for_metrics = np.vstack(all_probabilities_for_metrics)

    precision = precision_score(final_labels_for_metrics, final_predictions_for_metrics, average='weighted', zero_division=0)
    recall = recall_score(final_labels_for_metrics, final_predictions_for_metrics, average='weighted', zero_division=0)
    f1 = f1_score(final_labels_for_metrics, final_predictions_for_metrics, average='weighted', zero_division=0)
    auc = roc_auc_score(final_labels_for_metrics, final_probabilities_for_metrics, average='weighted')
    hamming_score = calculate_hamming_score_sklearn(final_labels_for_metrics, final_predictions_for_metrics)

    print(f"Epoch {epoch_num} Train - Loss: {avg_loss:.4f} | Acc: {accuracy:.4f}")

    return {
        'loss': avg_loss,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'auc': auc,
        'hamming_score': hamming_score
    }

def validate(model, val_loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct_samples = 0
    total_samples = 0

    all_predictions_for_metrics = []
    all_labels_for_metrics = []
    all_probabilities_for_metrics = [] # for AUC

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            probabilities = torch.sigmoid(outputs)
            predicted = (probabilities > 0.5).float()
            correct_samples += (predicted == labels).all(dim=1).sum().item()
            total_samples += labels.size(0)

            all_predictions_for_metrics.append(predicted.cpu().numpy())
            all_labels_for_metrics.append(labels.cpu().numpy())
            all_probabilities_for_metrics.append(probabilities.cpu().detach().numpy())

    avg_loss = running_loss / len(val_loader)
    accuracy = correct_samples / total_samples if total_samples > 0 else 0.0

    # calculate other metrics using collected predictions/labels
    final_predictions_for_metrics = np.vstack(all_predictions_for_metrics)
    final_labels_for_metrics = np.vstack(all_labels_for_metrics)
    final_probabilities_for_metrics = np.vstack(all_probabilities_for_metrics)

    precision = precision_score(final_labels_for_metrics, final_predictions_for_metrics, average='weighted', zero_division=0)
    recall = recall_score(final_labels_for_metrics, final_predictions_for_metrics, average='weighted', zero_division=0)
    f1 = f1_score(final_labels_for_metrics, final_predictions_for_metrics, average='weighted', zero_division=0)
    auc = roc_auc_score(final_labels_for_metrics, final_probabilities_for_metrics, average='weighted')
    hamming_score = calculate_hamming_score_sklearn(final_labels_for_metrics, final_predictions_for_metrics)

    return {
        'loss': avg_loss,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'auc': auc,
        'hamming_score': hamming_score
    }

def evaluate_test_set(model, test_loader, device):
    model.eval()

    all_predictions = []
    all_labels = []
    all_probabilities = [] # for AUC

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            probabilities = torch.sigmoid(outputs)
            predicted = (probabilities > 0.5).float()

            all_predictions.append(predicted.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            all_probabilities.append(probabilities.cpu().detach().numpy())

    all_predictions = np.vstack(all_predictions)
    all_labels = np.vstack(all_labels)
    all_probabilities = np.vstack(all_probabilities)

    # calculate metrics
    precision = precision_score(all_labels, all_predictions, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_predictions, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_predictions, average='weighted', zero_division=0)
    accuracy = np.all(all_predictions == all_labels, axis=1).mean()
    auc = roc_auc_score(all_labels, all_probabilities, average='weighted')
    hamming_score = calculate_hamming_score_sklearn(all_labels, all_predictions)

    return {
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'accuracy': accuracy,
        'auc': auc,
        'hamming_score': hamming_score
    }

### Plotting Functions

In [ ]:
def plot_training_history(history, fold, save_dir='output/kfold'):
    os.makedirs(save_dir, exist_ok=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    epochs = range(1, len(history['train_loss']) + 1)

    # plot loss
    ax1.plot(epochs, history['train_loss'], label='Train Loss', marker='o')
    ax1.plot(epochs, history['val_loss'], label='Validation Loss', marker='s')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'Fold {fold} - Loss Evolution')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_xticks(epochs)

    # plot accuracy
    ax2.plot(epochs, history['train_acc'], label='Train Accuracy', marker='o')
    ax2.plot(epochs, history['val_acc'], label='Validation Accuracy', marker='s')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title(f'Fold {fold} - Accuracy Evolution')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_xticks(epochs)

    plt.tight_layout()
    plt.savefig(f'{save_dir}/fold_{fold}_training_history.png', dpi=300, bbox_inches='tight')
    plt.close()

#### Compute std and mean for validation / test metrics

In [ ]:
def analyze_and_save_kfold_results(fold_results, experiment_name="", output_filename=None):
    # calculate mean and std for validation metrics
    val_precision = [r['val_metrics']['precision'] for r in fold_results]
    val_recall = [r['val_metrics']['recall'] for r in fold_results]
    val_f1 = [r['val_metrics']['f1_score'] for r in fold_results]
    val_accuracy = [r['val_metrics']['accuracy'] for r in fold_results]
    val_auc = [r['val_metrics']['auc'] for r in fold_results]
    val_hamming_score = [r['val_metrics']['hamming_score'] for r in fold_results]

    # calculate mean and std for test metrics
    test_precision = [r['test_metrics']['precision'] for r in fold_results]
    test_recall = [r['test_metrics']['recall'] for r in fold_results]
    test_f1 = [r['test_metrics']['f1_score'] for r in fold_results]
    test_accuracy = [r['test_metrics']['accuracy'] for r in fold_results]
    test_auc = [r['test_metrics']['auc'] for r in fold_results]
    test_hamming_score = [r['test_metrics']['hamming_score'] for r in fold_results]

    # create results dataframe
    results_data = []

    for i, result in enumerate(fold_results, 1):
        results_data.append({
            'Fold': i,
            'Val_Precision': result['val_metrics']['precision'],
            'Val_Recall': result['val_metrics']['recall'],
            'Val_F1': result['val_metrics']['f1_score'],
            'Val_Accuracy': result['val_metrics']['accuracy'],
            'Val_AUC': result['val_metrics']['auc'],
            'Val_Hamming_Score': result['val_metrics']['hamming_score'],
            'Test_Precision': result['test_metrics']['precision'],
            'Test_Recall': result['test_metrics']['recall'],
            'Test_F1': result['test_metrics']['f1_score'],
            'Test_Accuracy': result['test_metrics']['accuracy'],
            'Test_AUC': result['test_metrics']['auc'],
            'Test_Hamming_Score': result['test_metrics']['hamming_score']
        })

    # add mean row
    results_data.append({
        'Fold': 'Mean',
        'Val_Precision': np.mean(val_precision),
        'Val_Recall': np.mean(val_recall),
        'Val_F1': np.mean(val_f1),
        'Val_Accuracy': np.mean(val_accuracy),
        'Val_AUC': np.mean(val_auc),
        'Val_Hamming_Score': np.mean(val_hamming_score),
        'Test_Precision': np.mean(test_precision),
        'Test_Recall': np.mean(test_recall),
        'Test_F1': np.mean(test_f1),
        'Test_Accuracy': np.mean(test_accuracy),
        'Test_AUC': np.mean(test_auc),
        'Test_Hamming_Score': np.mean(test_hamming_score)
    })

    # add std row
    results_data.append({
        'Fold': 'Std',
        'Val_Precision': np.std(val_precision),
        'Val_Recall': np.std(val_recall),
        'Val_F1': np.std(val_f1),
        'Val_Accuracy': np.std(val_accuracy),
        'Val_AUC': np.std(val_auc),
        'Val_Hamming_Score': np.std(val_hamming_score),
        'Test_Precision': np.std(test_precision),
        'Test_Recall': np.std(test_recall),
        'Test_F1': np.std(test_f1),
        'Test_Accuracy': np.std(test_accuracy),
        'Test_AUC': np.std(test_auc),
        'Test_Hamming_Score': np.std(test_hamming_score)
    })

    results_df = pd.DataFrame(results_data)

    # display results
    title = f" {experiment_name}" if experiment_name else ""
    print(f"\nValidation Set Metrics{title}:")
    print(results_df[['Fold', 'Val_Precision', 'Val_Recall', 'Val_F1', 'Val_Accuracy', 'Val_AUC', 'Val_Hamming_Score']].to_string(index=False))

    print(f"\nTest Set Metrics{title}:")
    print(results_df[['Fold', 'Test_Precision', 'Test_Recall', 'Test_F1', 'Test_Accuracy', 'Test_AUC', 'Test_Hamming_Score']].to_string(index=False))

    # save results to csv
    os.makedirs('output/results', exist_ok=True)
    if output_filename is None:
        output_filename = 'output/results/kfold_results.csv'
    results_df.to_csv(output_filename, index=False)

    return results_df

### K-Fold Cross-Validation Implementation

In [ ]:
import numpy as np

from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    ScaleIntensityd,
    NormalizeIntensityd,
    MedianSmoothd,
    RandAdjustContrastd,
    Resized,
    RandFlipd,
    RandRotated,
    RandZoomd,
    RandAffined,
    ToTensord,
)

In [ ]:
CONFIG = {
    'k_folds': 5,
    'batch_size': 64,
    'num_epochs': 3,
    'learning_rate': 0.001,
    'random_state': 42,
    'device': device,
    'train_csv': 'subdataset_train.csv',
    'test_csv': 'subdataset_test.csv',
    'train_img_dir': 'rsna-intracranial-hemorrhage-detection-png/train_images',
    'test_img_dir': 'rsna-intracranial-hemorrhage-detection-png/test_images',
    'unfreeze_layers': -1,
    'checkpoint_dir': 'model_checkpoints',
    'l2_reg_lambda': 0.0
}

In [ ]:
radius=1
spatial_size=(224, 224)
gamma = 1.5

train_transforms = Compose([
    LoadImaged(keys='image'),
    EnsureChannelFirstd(keys=('image')),
    Resized(keys='image', spatial_size=spatial_size),
    NormalizeIntensityd(keys='image'),
    RandAffined(keys='image', prob=0.5,
                translate_range=(10, 10),
                shear_range=(0.1, 0.1),
                padding_mode='zeros'),
    RandAdjustContrastd(keys='image', prob=0.3, gamma=(0.9, 1.1)),
    ToTensord(keys='image'),
])

validation_transforms = Compose([
    LoadImaged(keys='image'),
    EnsureChannelFirstd(keys=('image')),
    Resized(keys='image', spatial_size=spatial_size),
    NormalizeIntensityd(keys='image'),
    ToTensord(keys='image')
])

test_transforms = Compose([
    LoadImaged(keys='image'),
    EnsureChannelFirstd(keys=('image')),
    Resized(keys='image', spatial_size=spatial_size),
    NormalizeIntensityd(keys='image'),
    ToTensord(keys='image')
])

#### Load datasets

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

In [ ]:
test_dataset = RSNAHemorrhageDataset(
    csv_file=CONFIG['test_csv'],
    img_dir=CONFIG['test_img_dir'],
    transform=test_transforms
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=2
)
print(f"Test dataset: {len(test_dataset)} images")

In [ ]:
# initialize K-Fold
kfold = KFold(n_splits=CONFIG['k_folds'], shuffle=True, random_state=CONFIG['random_state'])

# storage for results
fold_results = []
fold_histories = []

# create a base dataset to get indices
train_dataset = RSNAHemorrhageDataset(
    csv_file=CONFIG['train_csv'],
    img_dir=CONFIG['train_img_dir'],
    transform=train_transforms
)
indices = list(range(len(train_dataset)))

### Consolidated Training Function

In [ ]:
def run_kfold_training(dataset, kfold, config, use_class_weights=False,
                        experiment_name="default", class_weights=None,
                        resume_training_from_checkpoint=False,
                        selected_folds=None):

    global fold_results   # list to store results from all folds across all experiments
    global fold_histories # list to store training histories from all folds across all experiments
    global test_loader    # access the globally defined test_loader

    # this list will store results for the current experiment run (e.g., unweighted_results, weighted_results)
    local_experiment_results = []

    indices = list(range(len(dataset)))

    os.makedirs(config['checkpoint_dir'], exist_ok=True)

    for fold, (train_idx, val_idx) in enumerate(kfold.split(indices), 1):
        if selected_folds is not None and fold not in selected_folds:
            continue # skip if the current fold is not in the selected_folds list

        print(f"\n{'='*60}")
        print(f"Fold {fold}/{config['k_folds']} - {experiment_name}")
        print(f"{'='*60}")

        # create data subsets
        train_subset = torch.utils.data.Subset(dataset, train_idx)
        val_subset = torch.utils.data.Subset(dataset, val_idx)

        # create data loaders
        train_loader = DataLoader(
            train_subset,
            batch_size=config['batch_size'],
            shuffle=True,
            num_workers=0
        )

        val_loader = DataLoader(
            val_subset,
            batch_size=config['batch_size'],
            shuffle=False,
            num_workers=0
        )

        # initialize model with unfreeze_layers and dropout_prob from config
        model = IntracranialHemorrhageEfficientNet(
            unfreeze_layers=config.get('unfreeze_layers', -1),
            dropout_prob=config.get('dropout_prob', 0.3)
        ).to(config['device'])

        # initialize optimizer with weight_decay for L2 regularization
        optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'],
                               weight_decay=config.get('l2_reg_lambda', 0.0))

        # setup loss function with or without weights
        if use_class_weights and class_weights is not None:
            criterion = nn.BCEWithLogitsLoss(
                pos_weight=class_weights.to(config['device'])
            )
        else:
            criterion = nn.BCEWithLogitsLoss()

        # check for checkpoint to resume training
        start_epoch = 1
        checkpoint_path = os.path.join(config['checkpoint_dir'], f'{experiment_name}_fold_{fold}_checkpoint.pth')

        if resume_training_from_checkpoint and os.path.exists(checkpoint_path):
            print(f"Loading checkpoint from {checkpoint_path}")

            checkpoint = torch.load(checkpoint_path, map_location=config['device'])
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch = checkpoint['epoch'] + 1

            print(f"Resuming training from epoch {start_epoch}")

        # training history
        history = {
            'train_loss': [],
            'train_acc': [],
            'val_loss': [],
            'val_acc': []
        }

        print(f"\nTraining Fold {fold}...")

        # training loop
        for epoch in range(start_epoch, config['num_epochs'] + 1):
            train_metrics = train_epoch(model, train_loader, criterion, optimizer, config['device'], epoch)
            val_metrics = validate(model, val_loader, criterion, config['device'])

            # store history
            history['train_loss'].append(train_metrics['loss'])
            history['train_acc'].append(train_metrics['accuracy'])
            history['val_loss'].append(val_metrics['loss'])
            history['val_acc'].append(val_metrics['accuracy'])

            # print progress
            if epoch % 5 == 0 or epoch == start_epoch:
                print(f"Epoch [{epoch}/{config['num_epochs']}] - "
                      f"Train Loss: {train_metrics['loss']:.4f}, Train Acc: {train_metrics['accuracy']:.4f} | "
                      f"Val Loss: {val_metrics['loss']:.4f}, Val Acc: {val_metrics['accuracy']:.4f}")

            # save checkpoint after each epoch
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_metrics['loss'],
                'history': history
            }, checkpoint_path)
            print(f"Checkpoint saved for Fold {fold}, Epoch {epoch} at {checkpoint_path}")

        # plot training history
        plot_training_history(history, fold, save_dir=f'output/{experiment_name}')
        fold_histories.append(history)

        # evaluate on validation set
        print(f"\nEvaluating Fold {fold} on validation set...")
        val_metrics_final = validate(model, val_loader, criterion, config['device'])

        # evaluate on test set (with test transforms)
        print(f"Evaluating Fold {fold} on test set...")
        test_metrics_final = evaluate_test_set(model, test_loader, config['device'])

        # store results
        result = {
            'fold': fold,
            'use_weights': use_class_weights,
            'history': history,
            'val_metrics': val_metrics_final,
            'test_metrics': test_metrics_final
        }

        fold_results.append(result)
        local_experiment_results.append(result)

        # print fold results
        print(f"\nFold {fold} Results:")
        print("Validation Set:")
        print(f"Precision: {val_metrics_final['precision']:.4f}")
        print(f"Recall: {val_metrics_final['recall']:.4f}")
        print(f"F1-Score: {val_metrics_final['f1_score']:.4f}")
        print(f"Accuracy: {val_metrics_final['accuracy']:.4f}")
        print(f"AUC: {val_metrics_final['auc']:.4f}")

        print("Test Set:")
        print(f"Precision: {test_metrics_final['precision']:.4f}")
        print(f"Recall: {test_metrics_final['recall']:.4f}")
        print(f"F1-Score: {test_metrics_final['f1_score']:.4f}")
        print(f"Accuracy: {test_metrics_final['accuracy']:.4f}")
        print(f"AUC: {test_metrics_final['auc']:.4f}")

    return local_experiment_results

#### Training without weights

In [ ]:
# K-Fold Cross-Validation without class weights
unweighted_results = run_kfold_training(
    dataset=train_dataset,
    kfold=kfold,
    config=CONFIG,
    use_class_weights=False,
    experiment_name="kfold_unweighted"
)

### Calculate and Display Final Results

In [ ]:
df_standard = analyze_and_save_kfold_results(
    fold_results,
    experiment_name="Unweighted",
    output_filename='output/results/kfold_results.csv'
)

### Create Summary Visualization

In [ ]:
# plot comparison across folds
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle('K-Fold Cross-Validation Results Summary', fontsize=16, fontweight='bold')

metrics = ['Precision', 'Recall', 'F1-Score', 'Accuracy']

val_data = [[metric_list[i] for i in range(CONFIG['k_folds'])] for metric_list in [val_precision, val_recall, val_f1, val_accuracy]]
test_data = [[metric_list[i] for i in range(CONFIG['k_folds'])] for metric_list in [test_precision, test_recall, test_f1, test_accuracy]]

for idx, (metric, val_vals, test_vals) in enumerate(zip(metrics, val_data, test_data)):
    row = idx // 2
    col = idx % 2
    ax = axes[row, col]

    x = np.arange(1, CONFIG['k_folds'] + 1)
    width = 0.35

    ax.bar(x - width/2, val_vals, width, label='Validation', alpha=0.8)
    ax.bar(x + width/2, test_vals, width, label='Test', alpha=0.8)

    ax.axhline(y=np.mean(val_vals), color='blue', linestyle='--', alpha=0.5, label='Val Mean')
    ax.axhline(y=np.mean(test_vals), color='orange', linestyle='--', alpha=0.5, label='Test Mean')

    ax.set_xlabel('Fold')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} Across Folds')
    ax.set_xticks(x)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('output/results/kfold_summary.png', dpi=300, bbox_inches='tight')
plt.show()

## Balancing Classes

### 1. Weighted loss function

In [ ]:
def calculate_class_distribution(dataset, indices, label_columns):
  total_count = len(indices)
  positive_counts = {label: 0 for label in label_columns}

  for idx in indices:
      _, label = dataset[idx]
      for i, label_name in enumerate(label_columns):
          if label[i] == 1:
              positive_counts[label_name] += 1

  return positive_counts

In [ ]:
def compute_class_weights(dataset, indices, label_columns):
  positive_counts = calculate_class_distribution(dataset, indices, label_columns)
  total_count = len(indices)
  negative_counts = {label: total_count - count for label, count in positive_counts.items()}

  class_weights_dict = {}
  distribution_info = {}

  for label in label_columns:
    positive_count = positive_counts[label]
    negative_count = negative_counts[label]

    # calculate weight
    pos_weight_val = negative_count / (positive_count + 1e-5)
    class_weights_dict[label] = pos_weight_val

    distribution_info[label] = {
        'positive_count': positive_count,
        'negative_count': negative_count,
        'weight': pos_weight_val
    }

  return class_weights_dict, distribution_info

In [ ]:
def plot_class_distribution_with_weights(distribution_info, label_columns, save_path):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    x = np.arange(len(label_columns))
    width = 0.35

    positive_counts = [distribution_info[label]['positive_count'] for label in label_columns]
    negative_counts = [distribution_info[label]['negative_count'] for label in label_columns]

    ax1.bar(x - width/2, positive_counts, width, label='Positive',
            color='lightcoral', alpha=0.8)
    ax1.bar(x + width/2, negative_counts, width, label='Negative',
            color='skyblue', alpha=0.8)

    ax1.set_xlabel('Hemorrhage Type', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Number of Instances', fontsize=12, fontweight='bold')
    ax1.set_title('Class Distribution', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(label_columns, rotation=45, ha='right')
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close()

In [ ]:
_, distribution_info = compute_class_weights(train_dataset, indices, train_dataset.label_columns)
print("Distribution Info:")
print(distribution_info)

In [ ]:
plot_class_distribution_with_weights(distribution_info, train_dataset.label_columns, 'class_distribution_with_weights.png')

In [ ]:
def plot_comparison_weighted_vs_unweighted(weighted_results, unweighted_results,
                                          fold, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    epochs = range(1, len(weighted_results['history']['train_loss']) + 1)

    metrics = ['precision', 'recall', 'f1_score', 'accuracy']
    metric_names = ['Precision', 'Recall', 'F1-Score', 'Accuracy']

    unweighted_vals = [unweighted_results['test_metrics'][m] for m in metrics]
    weighted_vals = [weighted_results['test_metrics'][m] for m in metrics]

    x = np.arange(len(metrics))
    width = 0.35

    axes[1, 0].bar(x - width/2, unweighted_vals, width, label='Unweighted',
                   color='skyblue', alpha=0.8)
    axes[1, 0].bar(x + width/2, weighted_vals, width, label='Weighted',
                   color='lightcoral', alpha=0.8)
    axes[1, 0].set_xlabel('Metrics', fontsize=12, fontweight='bold')
    axes[1, 0].set_ylabel('Score', fontsize=12, fontweight='bold')
    axes[1, 0].set_title(f'Fold {fold} - Test Metrics', fontsize=14, fontweight='bold')
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels(metric_names)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3, axis='y')

    fig.delaxes(axes[1, 1])

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close()

#### Training with weighted loss

In [ ]:
# compute class weights from training data
weights_dict, _ = compute_class_weights(train_dataset, indices, train_dataset.label_columns)

# convert the dictionary of weights to a tensor in the correct order
weights_list = [weights_dict[label] for label in train_dataset.label_columns]
weights_tensor = torch.tensor(weights_list, dtype=torch.float32)

weighted_results = run_kfold_training(
    dataset=train_dataset,
    kfold=kfold,
    config=CONFIG,
    use_class_weights=True,
    experiment_name="kfold_weighted",
    class_weights=weights_tensor
)


In [ ]:
df_weighted = analyze_and_save_kfold_results(
    weighted_results,
    experiment_name="(Weighted Loss)",
    output_filename='output/results/kfold_results_weighted.csv'
)

In [ ]:
os.makedirs('output/results/comparison', exist_ok=True)

for fold in range(1, CONFIG['k_folds'] + 1):
    print(f"Fold {fold} - comparison with and without weights")

    # get results for this fold
    unweighted_fold = unweighted_results[fold - 1]
    weighted_fold = weighted_results[fold - 1]

    # create comparison plot
    plot_comparison_weighted_vs_unweighted(
        weighted_fold,
        unweighted_fold,
        fold,
        f'output/results/comparison/fold_{fold}_comparison.png'
    )

### 2. Oversampling

#### 1. Duplicate samples without augmentation

In [ ]:
def compute_class_sizes(dataset, label_columns):
  class_counts = {}

  for _, label in dataset:
    for i, label_name in enumerate(label_columns):
      if label_name != "any" and label[i] == 1:
        class_counts[label_name] = class_counts.get(label_name, 0) + 1

  return class_counts

In [ ]:
class_counts = compute_class_sizes(train_dataset, train_dataset.label_columns)

majority_class = max(class_counts, key=class_counts.get)
majority_count = class_counts[majority_class]

print(f"Majority class: {majority_class}")
print(f"Majority count: {majority_count}")

In [ ]:
def extract_indices_with_target_class(dataset, indices, target_class):
    target_idx = dataset.label_columns.index(target_class)
    target_indices = []

    for idx in indices:
        _, label = dataset[idx]
        if label[target_idx] == 1:
            target_indices.append(idx)

    return target_indices

In [ ]:
output_base_dir = 'output/oversampled_dataset'
output_img_dir = os.path.join(output_base_dir, 'rsna-intracranial-hemorrhage-detection-png/train_images')
os.makedirs(output_img_dir, exist_ok=True)

In [ ]:
all_csv_rows = []
duplicate_counter = 0

In [ ]:
import shutil

def copy_already_existing_train_images(indices, label_columns):
  global all_csv_rows
  for idx in indices:
    img_id = train_dataset.image_ids[idx]

    img_filename = f"{img_id}_frame0.png"

    # copy original image
    src_path = os.path.join(CONFIG['train_img_dir'], img_filename)
    dst_path = os.path.join(output_img_dir, img_filename)

    if os.path.exists(src_path):
        shutil.copy2(src_path, dst_path)
        print(f"Copied: {img_filename}")

    # get labels and add to csv
    _, labels = train_dataset[idx]
    for i, label_name in enumerate(label_columns):
        all_csv_rows.append({
            'ID': f"{img_id}_{label_name}",
            'Label': int(labels[i])
        })


In [ ]:
indices = list(range(len(train_dataset)))
copy_already_existing_train_images(indices, train_dataset.label_columns)

In [ ]:
# check how many images are in "output/oversampled_data/rsna/train_images"

img_path = 'output/oversampled_dataset/rsna-intracranial-hemorrhage-detection-png/train_images'
img_files = os.listdir(img_path)
print(f"Number of images in '{img_path}': {len(img_files)}")

# print the len of all_csv_rows
print(f"Number of rows in all_csv_rows: {len(all_csv_rows)}")

In [ ]:
import pandas as pd

original_csv_path = 'subdataset_train.csv'
df_original = pd.read_csv(original_csv_path)
all_csv_rows = df_original.to_dict('records')

print(f"Original CSV entries: {len(all_csv_rows)}")

In [ ]:
def duplicate_target_class_images(random_indices, all_dataset_label_columns):
    global all_csv_rows

    duplicate_counter = 0

    for idx in random_indices:
        orig_img_id = train_dataset.image_ids[idx]

        # create unique duplicate id
        duplicate_counter += 1
        dup_img_id = f"{orig_img_id}_dup{duplicate_counter}"

        # construct duplicated image filename
        orig_img_filename = f"{orig_img_id}_frame0.png"
        dup_img_filename = f"{dup_img_id}_frame0.png"

        src_path = os.path.join(CONFIG['train_img_dir'], orig_img_filename)
        dst_path = os.path.join(output_img_dir, dup_img_filename)

        if os.path.exists(src_path):
            shutil.copy2(src_path, dst_path)

        # get labels for this sample
        _, labels = train_dataset[idx]

        # add duplicate to csv with all label columns
        for i, label_name in enumerate(all_dataset_label_columns):
            all_csv_rows.append({
                'ID': f"{dup_img_id}_{label_name}",
                'Label': int(labels[i])
            })

In [ ]:
label_columns = ['epidural','intraparenchymal','intraventricular','subarachnoid','subdural']

total_duplicates = 0

for label in label_columns:
  print(f"Class: {label}")
  print(f"No of samples: {class_counts[label]}")

  indices_with_target_class = extract_indices_with_target_class(train_dataset, indices, label)

  current_count = len(indices_with_target_class)
  no_of_duplicated_samples = majority_count - current_count

  print(f"Current count: {current_count}")
  print(f"Target count: {majority_count}")
  print(f"Duplicates needed: {no_of_duplicated_samples}")
  print("\n")

  if no_of_duplicated_samples == 0:
    continue

  random_indices = np.random.choice(
      indices_with_target_class,
      no_of_duplicated_samples,
      replace=False
  )

  duplicate_target_class_images(random_indices, train_dataset.label_columns)
  total_duplicates += no_of_duplicated_samples

print(f"\nTotal images duplicated: {total_duplicates}")

In [ ]:
img_path = 'output/oversampled_dataset/rsna-intracranial-hemorrhage-detection-png/train_images'
img_files = os.listdir(img_path)
print(f"Number of images in '{img_path}': {len(img_files)}")

In [ ]:
df_oversampled = pd.DataFrame(all_csv_rows)
csv_output_path = os.path.join(output_base_dir, 'subdataset_train.csv')
df_oversampled.to_csv(csv_output_path, index=False)z

In [ ]:
oversampled_train_dataset = RSNAHemorrhageDataset(
    csv_file=os.path.join(output_base_dir, 'subdataset_train.csv'),
    img_dir=output_img_dir,
    transform=train_transforms
)
oversampled_indices = list(range(len(oversampled_train_dataset)))

print("Calculating class distribution for the oversampled dataset...")
_, oversampled_distribution_info = compute_class_weights(
    oversampled_train_dataset,
    oversampled_indices,
    oversampled_train_dataset.label_columns
)

print("Oversampled Distribution Info:")
print(oversampled_distribution_info)

# plot the new distribution
os.makedirs('output/results/oversampled', exist_ok=True)
plot_class_distribution_with_weights(
    oversampled_distribution_info,
    oversampled_train_dataset.label_columns,
    'output/results/oversampled/oversampled_class_distribution.png'
)

#### Training with oversampling and without weights



In [ ]:
def plot_kfold_comparison(results1, results2, label1, label2, comparison_title, output_filename, k_folds):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(comparison_title, fontsize=16, fontweight='bold')

    metrics = ['precision', 'recall', 'f1_score', 'accuracy']
    titles = ['Precision', 'Recall', 'F1-Score', 'Accuracy']

    for idx, (metric, title) in enumerate(zip(metrics, titles)):
        ax = axes[idx // 2, idx % 2]

        vals1 = [r['val_metrics'][metric] for r in results1]
        vals2 = [r['val_metrics'][metric] for r in results2]

        folds = list(range(1, k_folds + 1))

        ax.plot(folds, vals1, 'o-', label=label1, linewidth=2, markersize=8)
        ax.plot(folds, vals2, 's-', label=label2, linewidth=2, markersize=8)

        ax.set_xlabel('Fold', fontsize=12)
        ax.set_ylabel(title, fontsize=12)
        ax.set_title(f'{title} by Fold', fontsize=13, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_xticks(folds)

    plt.tight_layout()

    # create output directory if it doesn't exist
    os.makedirs('output/results/comparison', exist_ok=True)
    save_path = f'output/results/comparison/{output_filename}'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# K-Fold Cross-Validation with oversampled dataset but no class weights
oversampled_unweighted_results = run_kfold_training(
    dataset=oversampled_train_dataset,
    kfold=KFold(n_splits=CONFIG['k_folds'], shuffle=True, random_state=CONFIG['random_state']),
    config=CONFIG,
    use_class_weights=False,
    experiment_name="kfold_oversampled_unweighted"
)


In [ ]:
df_oversampled = analyze_and_save_kfold_results(
    oversampled_unweighted_results,
    experiment_name="(Oversampled + Unweighted Loss)",
    output_filename='output/results/kfold_oversampled_unweighted_results.csv'
)

#### Results Comparison: Oversampling vs No Oversampling

In [ ]:
plot_kfold_comparison(
    results1=unweighted_results,
    results2=oversampled_unweighted_results,
    label1='No Weights, No Oversampling',
    label2='Oversampled, No Weights',
    comparison_title='Performance Comparison: Unweighted vs Oversampled+Unweighted',
    output_filename='unweighted_vs_oversampled_unweighted.png',
    k_folds=CONFIG['k_folds']
)

#### 2. Duplicate samples but before use augumentations

In [ ]:
output_base_dir = 'output/oversampled_dataset_augumented'
output_img_dir = os.path.join(output_base_dir, 'rsna-intracranial-hemorrhage-detection-png/train_images')
os.makedirs(output_img_dir, exist_ok=True)

In [ ]:
import numpy as np
from PIL import Image

from monai.transforms import (
    Compose,
    LoadImage,
    EnsureChannelFirst,
    ScaleIntensity,
    RandAdjustContrast,
    Resize,
    RandFlip,
    RandRotate,
    RandZoom,
    RandAffined,
)

import torch
import shutil

In [1]:
def get_oversampling_augmentation_transforms(spatial_size=(224, 224)):
    return Compose([
        LoadImage(image_only=True, ensure_channel_first=True, reader='PILReader'),
        Resize(spatial_size=spatial_size),
        RandAffined(prob=0.7,
                   translate_range=(15, 15),
                   shear_range=(0.15, 0.15),
                   padding_mode='zeros',
                   mode='bilinear'),
        RandAdjustContrast(prob=0.5, gamma=(0.8, 1.2)),
        RandFlip(prob=0.7, spatial_axis=0),
        RandRotate(prob=0.7, range_x=np.pi/8, mode='bilinear', padding_mode='zeros'),
        RandZoom(prob=0.7, min_zoom=0.8, max_zoom=1.2, mode='bilinear'),
        ScaleIntensity(),
        lambda x: (x * 255).astype(np.uint8).squeeze(),
    ])

In [ ]:
def duplicate_target_class_images(random_indices, all_dataset_label_columns, image_augmentation_transform):
    global all_csv_rows

    duplicate_counter = 0

    for idx in random_indices:
        orig_img_id = train_dataset.image_ids[idx]

        # create unique duplicate id
        duplicate_counter += 1
        dup_img_id = f"{orig_img_id}_aug_dup{duplicate_counter}"

        # construct duplicated image filename
        orig_img_filename = f"{orig_img_id}_frame0.png"
        dup_img_filename = f"{dup_img_id}_frame0.png"

        src_path = os.path.join(CONFIG['train_img_dir'], orig_img_filename)
        dst_path = os.path.join(output_img_dir, dup_img_filename)

        if os.path.exists(src_path):
            try:
                augmented_image_array = image_augmentation_transform(src_path)

                # ensure it's a 2D array if it's (H, W) or (H, W, 1)
                if augmented_image_array.ndim == 3 and augmented_image_array.shape[0] == 1:
                    augmented_image_array = augmented_image_array.squeeze(0)

                # save the augmented image using PIL
                augmented_pil_img = Image.fromarray(augmented_image_array, mode='L')
                augmented_pil_img.save(dst_path)

            except Exception as e:
                print(f"Error augmenting/saving image {orig_img_id}: {e}")

        _, labels = train_dataset[idx]

        # add duplicate to csv with all label columns
        for i, label_name in enumerate(all_dataset_label_columns):
            all_csv_rows.append({
                'ID': f"{dup_img_id}_{label_name}",
                'Label': int(labels[i])
            })

#### Training with augmented oversampled data

In [ ]:
label_columns = ['epidural','intraparenchymal','intraventricular','subarachnoid','subdural']

# create augmented dataset by duplicating minority classes with augmentation
augmentation_transform = get_oversampling_augmentation_transforms(spatial_size=(224, 224))

all_csv_rows = []
duplicate_counter = 0

# majority class
class_counts = compute_class_sizes(train_dataset, train_dataset.label_columns)

majority_class = max(class_counts, key=class_counts.get)
majority_count = class_counts[majority_class]

# copy existing images first
copy_already_existing_train_images(indices, train_dataset.label_columns)

img_path = 'output/oversampled_dataset_augumented/rsna-intracranial-hemorrhage-detection-png/train_images'
img_files = os.listdir(img_path)

# duplicate minority classes with augmentation
for target_class in label_columns:
    target_indices = extract_indices_with_target_class(
        train_dataset,
        indices,
        target_class
    )

    # determine how many duplicates to create
    target_count = class_counts[target_class]
    num_duplicates_needed = majority_count - target_count

    # sample indices to duplicate
    random_sample_indices = np.random.choice(
        target_indices,
        num_duplicates_needed,
        replace=True
    )

    print(f"Creating {num_duplicates_needed} augmented samples for {target_class}")

    # duplicate with augmentation
    duplicate_target_class_images(
        random_sample_indices,
        train_dataset.label_columns,
        augmentation_transform
    )

# save augmented dataset csv
df_augmented = pd.DataFrame(all_csv_rows)
augmented_csv_path = os.path.join(output_base_dir, 'stage_2_train_augmented.csv')
df_augmented.to_csv(augmented_csv_path, index=False)

# load augmented dataset
augmented_train_dataset = RSNAHemorrhageDataset(
    csv_file=augmented_csv_path,
    img_dir=output_img_dir,
    transform=train_transforms
)

#### K-Fold training with augmented data and no weights

In [ ]:
# train with augmented dataset and without class weights
augmented_unweighted_results = run_kfold_training(
    dataset=augmented_train_dataset,
    kfold=KFold(n_splits=CONFIG['k_folds'], shuffle=True, random_state=CONFIG['random_state']),
    config=CONFIG,
    use_class_weights=False,
    experiment_name="kfold_augmented_unweighted"
)

In [ ]:
df_augmented = analyze_and_save_kfold_results(
    augmented_unweighted_results,
    experiment_name="(Augmented + Unweighted Loss)",
    output_filename='output/results/kfold_augmented_unweighted_results.csv')

In [ ]:
plot_kfold_comparison(
    results1=unweighted_results,
    results2=augmented_unweighted_results,
    label1='No Weights, No Oversampling',
    label2='Oversampled, Augmented, No Weights',
    comparison_title='Performance Comparison: Unweighted vs Oversampled+Unweighted+Augmented',
    output_filename='unweighted_vs_oversampled_augmented_unweighted.png',
    k_folds=CONFIG['k_folds']
)

### 1. Train with frozen backbone

In [ ]:
# clear previous results for a new run
fold_results = []
fold_histories = []

frozen_backbone_results = run_kfold_training(
    dataset=train_dataset,
    kfold=kfold,
    config=CONFIG,
    use_class_weights=False,
    experiment_name="train_frozen_backbone_5_epoches"
)

# analyze and save results for this specific run
df_fronzen_backbone_5_epoches = analyze_and_save_kfold_results(
    frozen_backbone_results,
    experiment_name=" (Frozen Backbone, 5 Epochs)",
    output_filename='output/results/train_frozen_backbone_5_epoches.csv'
)

### 2. Train with unfrozen backbone

In [ ]:
# clear previous results for a new run
fold_results = []
fold_histories = []

unfrozen_backbone_results = run_kfold_training(
    dataset=train_dataset,
    kfold=kfold,
    config=CONFIG,
    use_class_weights=False,
    experiment_name="train_unfrozen_backbone_5_epoches"
)

# analyze and save results for this specific run
df_unfronzen_backbone_5_epoches = analyze_and_save_kfold_results(
    unfrozen_backbone_results,
    experiment_name=" (Unfrozen Backbone, 5 Epochs)",
    output_filename='output/results/train_unfrozen_backbone_5_epoches.csv'
)

## Augmentations

In [ ]:
import numpy as np
import torch
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    ScaleIntensityd,
    Resized,
    RandFlipd,
    RandRotated,
    RandZoomd,
    RandAffined,
    RandAdjustContrastd,
    RandGaussianNoised,
    RandGaussianSmoothd,
    ToTensord,
)

### Define Hamming Score

In [ ]:
from sklearn.metrics import hamming_loss

def calculate_hamming_score_sklearn(y_true, y_pred):

    # convert to numpy if needed
    if torch.is_tensor(y_true):
        y_true = y_true.cpu().numpy()
    if torch.is_tensor(y_pred):
        y_pred = y_pred.cpu().numpy()

    # ensure binary predictions
    y_pred = (y_pred > 0.5).astype(int) if y_pred.dtype == float else y_pred

    # Hamming Score = 1 - Hamming Loss
    return 1.0 - hamming_loss(y_true, y_pred)

### Define 3 different sets of augmentations

In [ ]:
class AugmentationSets:
  def __init__(self, spatial_size=(224, 224), radius=1, gamma=1.5, base_transforms=None):
        self.spatial_size = spatial_size
        self.radius = radius
        self.gamma = gamma
        self.base_transforms = [
          LoadImaged(keys='image'),
          EnsureChannelFirstd(keys='image'),
          Resized(keys='image', spatial_size=self.spatial_size, mode='bilinear'),
          ScaleIntensityd(keys='image'),
        ]

  def get_augmentation_set_1(self):
    train_transforms = Compose(self.base_transforms + [
            RandRotated(keys='image', prob=0.3, range_x=0.1),
            RandFlipd(keys='image', prob=0.5, spatial_axis=0),
            RandAffined(keys='image', prob=0.3,
                      translate_range=(5, 5),
                      scale_range=(0.05, 0.05)),
            RandAdjustContrastd(keys='image', prob=0.3, gamma=(0.9, 1.1)),
            ToTensord(keys='image')
        ])

    val_transforms = Compose(self.base_transforms + [ToTensord(keys='image')])
    test_transforms = Compose(self.base_transforms + [ToTensord(keys='image')])

    return {
        'train': train_transforms,
        'val': val_transforms,
        'test': test_transforms,
        'name': 'Augmentation_Set_1'
    }

  def get_augmentation_set_2(self):
      train_transforms = Compose(self.base_transforms + [
          RandRotated(keys='image', prob=0.7, range_x=0.2),
          RandFlipd(keys='image', prob=0.5, spatial_axis=0),
          RandAffined(keys='image', prob=0.6,
                    translate_range=(15, 15),
                    scale_range=(0.15, 0.15),
                    shear_range=(0.15, 0.15),
                    padding_mode='zeros'),
          RandZoomd(keys='image', prob=0.4, min_zoom=0.85, max_zoom=1.15),
          RandAdjustContrastd(keys='image', prob=0.5, gamma=(0.8, 1.3)),
          RandGaussianNoised(keys='image', prob=0.3, std=0.02),
          RandGaussianSmoothd(keys='image', prob=0.2,
                            sigma_x=(0.5, 1.0),
                            sigma_y=(0.5, 1.0)),
      ])

      val_transforms = Compose(self.base_transforms + [ToTensord(keys='image')])
      test_transforms = Compose(self.base_transforms + [ToTensord(keys='image')])

      return {
          'train': train_transforms,
          'val': val_transforms,
          'test': test_transforms,
          'name': 'Augmentation_Set_2'
      }

  def get_augmentation_set_3(self):
    train_transforms = Compose(self.base_transforms + [
          RandRotated(keys='image', prob=0.5, range_x=0.15),
          RandFlipd(keys='image', prob=0.5, spatial_axis=0),
          RandAffined(keys='image', prob=0.5,
                    translate_range=(10, 10),
                    scale_range=(0.1, 0.1),
                    shear_range=(0.05, 0.05)),
          RandAdjustContrastd(keys='image', prob=0.5, gamma=(0.8, 1.2)),
          RandGaussianNoised(keys='image', prob=0.2, std=0.01),
          RandGaussianSmoothd(keys='image', prob=0.2,
                            sigma_x=(0.25, 0.75),
                            sigma_y=(0.25, 0.75)),

          ToTensord(keys='image')
    ])

    val_transforms = Compose(self.base_transforms + [ToTensord(keys='image')])
    test_transforms = Compose(self.base_transforms + [ToTensord(keys='image')])

    return {
        'train': train_transforms,
        'val': val_transforms,
        'test': test_transforms,
        'name': 'Augmentation_Set_3'
    }

  def get_all_sets(self):
    return [
        self.get_augmentation_set_1(),
        self.get_augmentation_set_2(),
        self.get_augmentation_set_3()
    ]


### Compare the results from all the augmentations

In [ ]:
def compare_augmentation_results(all_results):
    comparison_data = []

    for aug_name, results in all_results.items():
        # calculate average metrics across all folds
        val_metrics = {
            'precision': [],
            'recall': [],
            'f1_score': [],
            'accuracy': [],
            'auc': []
        }

        test_metrics = {
            'precision': [],
            'recall': [],
            'f1_score': [],
            'accuracy': [],
            'auc': []
        }

        for result in results:
            for metric in val_metrics.keys():
                if metric in result['val_metrics']:
                    val_metrics[metric].append(result['val_metrics'][metric])
                if metric in result['test_metrics']:
                    test_metrics[metric].append(result['test_metrics'][metric])

        # calculate means and stds
        comparison_data.append({
            'Augmentation Set': aug_name,
            'Val Precision': f"{np.mean(val_metrics['precision']):.4f} / {np.std(val_metrics['precision']):.4f}",
            'Val Recall': f"{np.mean(val_metrics['recall']):.4f} / {np.std(val_metrics['recall']):.4f}",
            'Val F1-Score': f"{np.mean(val_metrics['f1_score']):.4f} / {np.std(val_metrics['f1_score']):.4f}",
            'Val Accuracy': f"{np.mean(val_metrics['accuracy']):.4f} / {np.std(val_metrics['accuracy']):.4f}",
            'Val AUC': f"{np.mean(val_metrics['auc']):.4f} / {np.std(val_metrics['auc']):.4f}",
            'Test Precision': f"{np.mean(test_metrics['precision']):.4f} / {np.std(test_metrics['precision']):.4f}",
            'Test Recall': f"{np.mean(test_metrics['recall']):.4f} / {np.std(test_metrics['recall']):.4f}",
            'Test F1-Score': f"{np.mean(test_metrics['f1_score']):.4f} / {np.std(test_metrics['f1_score']):.4f}",
            'Test Accuracy': f"{np.mean(test_metrics['accuracy']):.4f} / {np.std(test_metrics['accuracy']):.4f}",
            'Test AUC': f"{np.mean(test_metrics['auc']):.4f} / {np.std(test_metrics['auc']):.4f}",
        })


    df = pd.DataFrame(comparison_data)
    return df

In [ ]:
aug_sets = AugmentationSets(spatial_size=(224, 224), radius=1, gamma=1.5)
all_augmentation_results = {}

### Train using the first set

In [ ]:
aug_set = aug_sets.get_augmentation_set_1()

print(f"Training with {aug_set['name']}")

# create dataset with the current augmentation set
train_dataset = RSNAHemorrhageDataset(
    csv_file=CONFIG['train_csv'],
    img_dir=CONFIG['train_img_dir'],
    transform=aug_set['train']
)

# update test dataset with new test transforms
test_dataset = RSNAHemorrhageDataset(
    csv_file=CONFIG['test_csv'],
    img_dir=CONFIG['test_img_dir'],
    transform=aug_set['test']
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=0
)

# run k-fold training, explicitly for folds 3 and 4
results = run_kfold_training(
    dataset=train_dataset,
    kfold=kfold,
    config=CONFIG,
    use_class_weights=False,
    experiment_name=aug_set['name'],
    selected_folds=[3, 4]
)

all_augmentation_results[aug_set['name']] = results

# analyze and save results for this augmentation set
df_results = analyze_and_save_kfold_results(
    results,
    experiment_name=aug_set['name'],
    output_filename=f"output/results/{aug_set['name']}_results.csv"
)

### Train using the second set

In [ ]:
aug_set = aug_sets.get_augmentation_set_2()

# create dataset with the current augmentation set
train_dataset = RSNAHemorrhageDataset(
    csv_file=CONFIG['train_csv'],
    img_dir=CONFIG['train_img_dir'],
    transform=aug_set['train']
)

# update test dataset with new test transforms
test_dataset = RSNAHemorrhageDataset(
    csv_file=CONFIG['test_csv'],
    img_dir=CONFIG['test_img_dir'],
    transform=aug_set['test']
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=0
)

# run k-fold training, explicitly for folds 3 and 4
results = run_kfold_training(
    dataset=train_dataset,
    kfold=kfold,
    config=CONFIG,
    use_class_weights=False,
    experiment_name=aug_set['name'],
    selected_folds=[3, 4]
)

all_augmentation_results[aug_set['name']] = results

# analyze and save results for this augmentation set
df_results = analyze_and_save_kfold_results(
    results,
    experiment_name=aug_set['name'],
    output_filename=f"output/results/{aug_set['name']}_results.csv"
)

### Train using the third set

In [ ]:
aug_set = aug_sets.get_augmentation_set_3()

# create dataset with the current augmentation set
train_dataset = RSNAHemorrhageDataset(
    csv_file=CONFIG['train_csv'],
    img_dir=CONFIG['train_img_dir'],
    transform=aug_set['train']
)

# update test dataset with new test transforms
test_dataset = RSNAHemorrhageDataset(
    csv_file=CONFIG['test_csv'],
    img_dir=CONFIG['test_img_dir'],
    transform=aug_set['test']
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=0
)

# run k-fold training, explicitly for folds 3 and 4
results = run_kfold_training(
    dataset=train_dataset,
    kfold=kfold,
    config=CONFIG,
    use_class_weights=False,
    experiment_name=aug_set['name'],
    selected_folds=[3, 4]
)

all_augmentation_results[aug_set['name']] = results

# analyze and save results for this augmentation set
df_results = analyze_and_save_kfold_results(
    results,
    experiment_name=aug_set['name'],
    output_filename=f"output/results/{aug_set['name']}_results.csv"
)

In [ ]:
# create and save comparative table
comparison_df = compare_augmentation_results(all_augmentation_results)
comparison_df.to_csv('output/results/augmentation_comparison.csv', index=False)

## Early Stopping + LR Scheduler

### Setup training components used for both methods

In [ ]:
def setup_training_components(train_dataset, kfold_splitter, config, selected_fold_num):
    indices = list(range(len(train_dataset)))
    train_loader = None
    val_loader = None

    for fold_num, (train_idx, val_idx) in enumerate(kfold_splitter.split(indices), 1):
        if fold_num == selected_fold_num:
            train_subset = torch.utils.data.Subset(train_dataset, train_idx)
            val_subset = torch.utils.data.Subset(train_dataset, val_idx)

            train_loader = DataLoader(
                train_subset,
                batch_size=config['batch_size'],
                shuffle=True,
                num_workers=0
            )

            val_loader = DataLoader(
                val_subset,
                batch_size=config['batch_size'],
                shuffle=False,
                num_workers=0
            )
            break

    print(f"Data loaders created for Fold {selected_fold_num}")
    print(f"Training samples: {len(train_subset)}")
    print(f"Validation samples: {len(val_subset)}")

    model = IntracranialHemorrhageEfficientNet(
        unfreeze_layers=config.get('unfreeze_layers', -1)
    ).to(config['device'])

    optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])

    return train_loader, val_loader, model, optimizer

In [ ]:
def create_training_results_csv_table(results, save_path, experiment_type):
    test_metrics = results['test_metrics']
    val_metrics = results.get('val_metrics_at_best_epoch') or results.get('val_metrics_at_last_epoch')

    data = {
        'Metric': [],
        'Value': []
    }

    # add validation metrics
    data['Metric'].extend([
        'Validation Loss', 'Validation Accuracy', 'Validation Precision',
        'Validation Recall', 'Validation F1-Score', 'Validation AUC', 'Validation Hamming Score'
    ])
    data['Value'].extend([
        f"{val_metrics['loss']:.4f}", f"{val_metrics['accuracy']:.4f}", f"{val_metrics['precision']:.4f}",
        f"{val_metrics['recall']:.4f}", f"{val_metrics['f1_score']:.4f}", f"{val_metrics['auc']:.4f}", f"{val_metrics['hamming_score']:.4f}"
    ])

    # add test metrics
    data['Metric'].extend([
        'Test Precision', 'Test Recall', 'Test F1-Score',
        'Test Accuracy', 'Test AUC', 'Test Hamming Score'
    ])
    data['Value'].extend([
        f"{test_metrics['precision']:.4f}", f"{test_metrics['recall']:.4f}", f"{test_metrics['f1_score']:.4f}",
        f"{test_metrics['accuracy']:.4f}", f"{test_metrics['auc']:.4f}", f"{test_metrics['hamming_score']:.4f}"
    ])

    # Add training metadata
    data['Metric'].extend(['Training Time (s)', 'Epochs Trained'])
    data['Value'].extend([
        f"{results['training_time']:.2f}",
        f"{results['epochs_trained']}"
    ])

    if experiment_type == 'EarlyStopping':
        data['Metric'].append('Best Epoch (Early Stop)')
        data['Value'].append(f"{results['best_epoch']}")

    df = pd.DataFrame(data)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    df.to_csv(save_path, index=False)
    print(f"\n{experiment_type} results table saved to {save_path}")

### 1. Early Stopping

In [ ]:
import time

In [ ]:
# configure a kfold splitter for this section
kfold_for_cerinta4 = KFold(n_splits=CONFIG['k_folds'], shuffle=True, random_state=CONFIG['random_state'])
selected_fold = 3

# setup components for Early Stopping
train_loader_es, val_loader_es, model_earlystop, optimizer_earlystop = setup_training_components(
    train_dataset, kfold_for_cerinta4, CONFIG, selected_fold
)

In [ ]:
class EarlyStopping:
  def __init__(self, patience=3, mode='min', delta=1e-4, path='checkpoint.pth'):
    self.patience = patience
    self.delta = delta
    self.mode = mode
    self.path = path

    self.counter = 0
    self.best_score = None
    self.early_stop = False
    self.best_epoch = 0
    self.best_model_state = None # to store the state_dict of the best model
    self.best_val_metrics = None # store the full val_metrics dict

  def __call__(self, current_score, epoch, model=None, val_metrics_dict=None):
    if self.best_score is None:
      self.best_score = current_score
      self.best_epoch = epoch
      self.best_val_metrics = val_metrics_dict
      if model is not None:
          self.save_checkpoint(model, current_score, epoch)
      print(f'Initial score set to {current_score:.6f}')
      return False

    improved = False
    if self.mode == 'min':
      if current_score < self.best_score + self.delta:
        improved = True

    elif self.mode == 'max':
      if current_score > self.best_score + self.delta:
        improved = True

    if improved:
        print(f'Validation metric improved: {self.best_score:.6f} -> {current_score:.6f}')
        self.counter = 0
        self.best_score = current_score
        self.best_epoch = epoch
        self.best_val_metrics = val_metrics_dict
        if model is not None:
            self.save_checkpoint(model, current_score, epoch)
        return False

    self.counter += 1
    print(f'Validation metric did not improve. Counter: {self.counter}/{self.patience}')
    if self.counter >= self.patience:
      self.early_stop = True
      return True

    return False

  def save_checkpoint(self, model, score, epoch):
      '''Saves model when validation metric improves.'''
      self.best_model_state = model.state_dict()
      torch.save({
          'epoch': epoch,
          'model_state_dict': model.state_dict(),
          'best_score': score,
          'best_val_metrics': self.best_val_metrics # save the metrics along with the model
      }, self.path)
      print(f'Model checkpoint saved to {self.path} (score: {score:.4f}, epoch: {epoch})')

  def get_best_epoch(self):
    return self.best_epoch

  def get_best_val_metrics(self):
      return self.best_val_metrics

In [ ]:
def train_with_early_stopping(model, train_loader, val_loader, test_loader,
                               criterion, optimizer, device, num_epochs,
                               early_stopping=None, experiment_name="EarlyStopping"):
  history = {
      'train_loss': [],
      'train_acc': [],
      'val_loss': [],
      'val_accuracy': [],
      'val_precision': [],
      'val_recall': [],
      'val_f1_score': [],
      'val_auc': [],
      'val_hamming_score': []
  }

  start_time = time.time()
  best_val_metrics_at_stop = None

  epoch = 1

  while True:
    print(f"Epoch {epoch}/{num_epochs} - {experiment_name}")

    train_metrics = train_epoch(model, train_loader, criterion, optimizer, device, epoch)
    val_metrics = validate(model, val_loader, criterion, device)

    history['train_loss'].append(train_metrics['loss'])
    history['train_acc'].append(train_metrics['accuracy'])
    history['val_loss'].append(val_metrics['loss'])
    history['val_accuracy'].append(val_metrics['accuracy'])
    history['val_precision'].append(val_metrics['precision'])
    history['val_recall'].append(val_metrics['recall'])
    history['val_f1_score'].append(val_metrics['f1_score'])
    history['val_auc'].append(val_metrics['auc'])
    history['val_hamming_score'].append(val_metrics['hamming_score'])

    print(f"Train Loss: {train_metrics['loss']:.4f} | Train Acc: {train_metrics['accuracy']:.4f} | Val Loss: {val_metrics['loss']:.4f} | Val Accuracy: {val_metrics['accuracy']:.4f}")

    if early_stopping is not None:
      if early_stopping(val_metrics['loss'], epoch, model, val_metrics_dict=val_metrics):
        print("Early stopping triggered")
        break

    epoch += 1

  # calculate training time
  training_time = time.time() - start_time
  print(f"Training time: {training_time:.2f} seconds")

  # load the best model found by early stopping if it was used
  if early_stopping and early_stopping.best_model_state:
      print(f"Loading best model from epoch {early_stopping.best_epoch} with validation loss {early_stopping.best_score:.4f}")
      model.load_state_dict(early_stopping.best_model_state)
      best_val_metrics_at_stop = early_stopping.get_best_val_metrics()
  else:
      # if no early stopping, or early stopping didn't save a best model,
      # use the metrics from the last epoch
      best_val_metrics_at_stop = val_metrics

  test_metrics = evaluate_test_set(model, test_loader, device)

  print(f"\nTest Results:")
  print(f"Precision: {test_metrics['precision']:.4f}")
  print(f"Recall: {test_metrics['recall']:.4f}")
  print(f"F1-Score: {test_metrics['f1_score']:.4f}")
  print(f"Accuracy: {test_metrics['accuracy']:.4f}")
  print(f"AUC: {test_metrics.get('auc', 0):.4f}")
  print(f"Epochs Trained: {len(history['train_loss'])}")
  print(f"Best Epoch: {early_stopping.best_epoch if early_stopping else num_epochs}")

  return {
      'history': history,
      'test_metrics': test_metrics,
      'val_metrics_at_best_epoch': best_val_metrics_at_stop,
      'training_time': training_time,
      'epochs_trained': len(history['train_loss']),
      'best_epoch': early_stopping.best_epoch if early_stopping else num_epochs
  }

In [ ]:
os.makedirs('output/cerinta4', exist_ok=True)
criterion=nn.BCEWithLogitsLoss()

# create early stopping object
early_stopping = EarlyStopping(
    patience=3,
    delta=1e-4,
    mode='min',
    path='output/cerinta4/early_stopping_best_model_fold3.pth'
)

earlystop_results = train_with_early_stopping(
    model_earlystop,
    train_loader_es,
    val_loader_es,
    test_loader,
    criterion,
    optimizer_earlystop,
    CONFIG['device'],
    CONFIG['num_epochs'],
    early_stopping=early_stopping,
    experiment_name="Early Stopping"
)

In [ ]:
# create visualizations and comparison table
plot_training_history(earlystop_results['history'], fold=selected_fold, save_dir='output/cerinta4')
create_training_results_csv_table(earlystop_results, save_path=f'output/cerinta4/fold_{selected_fold}/early_stopping_comparison.csv', experiment_type='EarlyStopping')

### 2. Learning Scheduler

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
import time
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
def train_with_steplr_schdelurer(model, train_loader, val_loader, test_loader, criterion, optimizer,
    num_epochs, device, fold=4, save_dir='output/cerinta4', step_size=3, gamma=0.1):

  scheduler = StepLR(optimizer, step_size=step_size, gamma=gamma)

  history = {
      'train_loss': [],
      'train_acc': [],
      'val_loss': [],
      'val_accuracy': [],
      'val_precision': [],
      'val_recall': [],
      'val_f1_score': [],
      'val_auc': [],
      'val_hamming_score': []
  }

  start_time = time.time()

  for epoch in range(1, num_epochs + 1):
    print(f"Epoch {epoch}/{num_epochs} - StepLR Scheduler")

    train_metrics = train_epoch(model, train_loader, criterion, optimizer, device, epoch)
    val_metrics = validate(model, val_loader, criterion, device)

    history['train_loss'].append(train_metrics['loss'])
    history['train_acc'].append(train_metrics['accuracy'])
    history['val_loss'].append(val_metrics['loss'])
    history['val_accuracy'].append(val_metrics['accuracy'])
    history['val_precision'].append(val_metrics['precision'])
    history['val_recall'].append(val_metrics['recall'])
    history['val_f1_score'].append(val_metrics['f1_score'])
    history['val_auc'].append(val_metrics['auc'])
    history['val_hamming_score'].append(val_metrics['hamming_score'])

    print(f"Train Loss: {train_metrics['loss']:.4f} | Train Acc: {train_metrics['accuracy']:.4f} | Val Loss: {val_metrics['loss']:.4f} | Val Accuracy: {val_metrics['accuracy']:.4f}")

    scheduler.step() # update learning rate
    print(f"Learning rate updated to: {scheduler.get_last_lr()[0]:.6f}")

  training_time = time.time() - start_time
  test_metrics = evaluate_test_set(model, test_loader, device)

  print(f"\nTest Results:")
  print(f"Precision: {test_metrics['precision']:.4f}")
  print(f"Recall: {test_metrics['recall']:.4f}")
  print(f"F1-Score: {test_metrics['f1_score']:.4f}")
  print(f"Accuracy: {test_metrics['accuracy']:.4f}")
  print(f"AUC: {test_metrics.get('auc', 0):.4f}")

  return {
      'history': history,
      'test_metrics': test_metrics,
      'val_metrics_at_last_epoch': val_metrics,
      'training_time': training_time,
      'epochs_trained': num_epochs
  }

In [ ]:
# setup components for StepLR Scheduler
train_loader_lr, val_loader_lr, model_steplr, optimizer_steplr = setup_training_components(
    train_dataset, kfold_for_cerinta4, CONFIG, selected_fold
)

In [ ]:
num_epochs_scheduler = 10

steplr_scheduler_results = train_with_steplr_schdelurer(
    model_steplr,
    train_loader_lr,
    val_loader_lr,
    test_loader,
    criterion,
    optimizer_steplr,
    num_epochs_scheduler,
    CONFIG['device'],
    fold=selected_fold
)

In [ ]:
plot_training_history(steplr_scheduler_results['history'], fold=selected_fold, save_dir='output/cerinta4')
create_training_results_csv_table(steplr_scheduler_results, save_path=f'output/cerinta4/fold_{selected_fold}/steplr_scheduler_comparison.csv', experiment_type='StepLRScheduler')

## Ablation Study

### 1. Train using batch_size = 16, fold = 3, num_epoches = 3

In [ ]:
results = run_kfold_training(
    dataset=train_dataset,
    kfold=kfold,
    config=CONFIG,
    use_class_weights=False,
    experiment_name="train_using_batch_size=16",
    selected_folds=[3]
)

df_results = analyze_and_save_kfold_results(
    results,
    experiment_name="train_using_batch_size=16",
    output_filename='output/results/"train_using_batch_size=16_results.csv'
)

### 2. Train using batch_size = 32, fold = 3, num_epoches = 3

In [ ]:
results = run_kfold_training(
    dataset=train_dataset,
    kfold=kfold,
    config=CONFIG,
    use_class_weights=False,
    experiment_name="train_using_batch_size=32",
    selected_folds=[3]
)

df_results = analyze_and_save_kfold_results(
    results,
    experiment_name="train_using_batch_size=32",
    output_filename='output/results/"train_using_batch_size=32_results.csv'
)

### 3. Train using batch_size = 64, fold = 3, num_epoches = 3

In [ ]:
results = run_kfold_training(
    dataset=train_dataset,
    kfold=kfold,
    config=CONFIG,
    use_class_weights=False,
    experiment_name="train_using_batch_size=64",
    selected_folds=[3]
)

df_results = analyze_and_save_kfold_results(
    results,
    experiment_name="train_using_batch_size=64",
    output_filename='output/results/"train_using_batch_size=64_results.csv'
)

### 4. Train using fold = 3, num_epoches = 2, optimizer = Adam

In [ ]:
import copy

CONFIG_C5_4 = copy.deepcopy(CONFIG)
CONFIG_C5_4['num_epochs'] = 2

In [ ]:
results_c5_4 = run_kfold_training(
    dataset=train_dataset,
    kfold=kfold,
    config=CONFIG_C5_4,
    use_class_weights=False,
    experiment_name="c5_4_fold3_epochs2_adam",
    selected_folds=[3]
)

df_results_c5_4 = analyze_and_save_kfold_results(
    results_c5_4,
    experiment_name="Cerinta 5.4 (Fold 3, 2 Epochs, Adam)",
    output_filename='output/results/c5_4_adam_fold3_epochs2_results.csv'
)

### 5. Train using fold = 3, num_epoches = 2, optimizer = SGD

In [ ]:
import copy
import os
import torch.optim as optim
from sklearn.model_selection import KFold

selected_fold = 3

local_experiment_results = []
indices = list(range(len(train_dataset)))

# initialize KFold splitter
kfold_splitter = KFold(n_splits=CONFIG_C5_4['k_folds'], shuffle=True, random_state=CONFIG_C5_4['random_state'])

# iterate through kfold splits to find the selected fold
for fold_num, (train_idx, val_idx) in enumerate(kfold_splitter.split(indices), 1):
    if fold_num == selected_fold:
        print(f"\n{'='*60}")
        print(f"Fold {fold_num}/{CONFIG_C5_4['k_folds']} - c5_5_fold3_epochs2_sgd")
        print(f"{'='*60}")

        # create data subsets
        train_subset = torch.utils.data.Subset(train_dataset, train_idx)
        val_subset = torch.utils.data.Subset(train_dataset, val_idx)

        # create data loaders
        train_loader = DataLoader(
            train_subset,
            batch_size=CONFIG_C5_4['batch_size'],
            shuffle=True,
            num_workers=0
        )

        val_loader = DataLoader(
            val_subset,
            batch_size=CONFIG_C5_4['batch_size'],
            shuffle=False,
            num_workers=0
        )

        # initialize model
        model = IntracranialHemorrhageEfficientNet(unfreeze_layers=CONFIG_C5_4.get('unfreeze_layers', -1)).to(CONFIG_C5_4['device'])
        optimizer = optim.SGD(model.parameters(), lr=CONFIG_C5_4['learning_rate']) # use SGD optimizer

        # setup loss function
        criterion = nn.BCEWithLogitsLoss()

        # training history
        history = {
            'train_loss': [],
            'train_acc': [],
            'val_loss': [],
            'val_acc': []
        }

        print(f"\nTraining Fold {fold_num}...")

        # training loop
        for epoch in range(1, CONFIG_C5_4['num_epochs'] + 1):
            train_metrics = train_epoch(model, train_loader, criterion, optimizer, CONFIG_C5_4['device'], epoch)
            val_metrics = validate(model, val_loader, criterion, CONFIG_C5_4['device'])

            # store history
            history['train_loss'].append(train_metrics['loss'])
            history['train_acc'].append(train_metrics['accuracy'])
            history['val_loss'].append(val_metrics['loss'])
            history['val_acc'].append(val_metrics['accuracy'])

            print(f"Epoch [{epoch}/{CONFIG_C5_4['num_epochs']}] - "
                  f"Train Loss: {train_metrics['loss']:.4f}, Train Acc: {train_metrics['accuracy']:.4f} | "
                  f"Val Loss: {val_metrics['loss']:.4f}, Val Acc: {val_metrics['accuracy']:.4f}")

        # plot training history
        os.makedirs(f'output/c5_5_fold3_epochs2_sgd', exist_ok=True)
        plot_training_history(history, fold_num, save_dir=f'output/c5_5_fold3_epochs2_sgd')

        # evaluate on validation set
        print(f"\nEvaluating Fold {fold_num} on validation set...")
        val_metrics_final = validate(model, val_loader, criterion, CONFIG_C5_4['device'])

        # evaluate on test set
        print(f"Evaluating Fold {fold_num} on test set...")
        test_metrics_final = evaluate_test_set(model, test_loader, CONFIG_C5_4['device'])

        # store results
        result = {
            'fold': fold_num,
            'use_weights': False,
            'history': history,
            'val_metrics': val_metrics_final,
            'test_metrics': test_metrics_final
        }

        local_experiment_results.append(result)

        # print fold results
        print(f"\nFold {fold_num} Results:")
        print("Validation Set:")
        print(f"Precision: {val_metrics_final['precision']:.4f}")
        print(f"Recall: {val_metrics_final['recall']:.4f}")
        print(f"F1-Score: {val_metrics_final['f1_score']:.4f}")
        print(f"Accuracy: {val_metrics_final['accuracy']:.4f}")
        print(f"AUC: {val_metrics_final['auc']:.4f}")

        print("Test Set:")
        print(f"Precision: {test_metrics_final['precision']:.4f}")
        print(f"Recall: {test_metrics_final['recall']:.4f}")
        print(f"F1-Score: {test_metrics_final['f1_score']:.4f}")
        print(f"Accuracy: {test_metrics_final['accuracy']:.4f}")
        print(f"AUC: {test_metrics_final['auc']:.4f}")

        break

df_results_c5_5 = analyze_and_save_kfold_results(
    local_experiment_results,
    experiment_name="Cerinta 5.5 (Fold 3, 2 Epochs, SGD)",
    output_filename='output/results/c5_5_sgd_fold3_epochs2_results.csv'
)

### 6. Dropout

In [ ]:
import copy

CONFIG_DROPOUT = copy.deepcopy(CONFIG)
CONFIG_DROPOUT['dropout_prob'] = 0.3
CONFIG_DROPOUT['num_epochs'] = 2

dropout_results = run_kfold_training(
    dataset=train_dataset,
    kfold=kfold,
    config=CONFIG_DROPOUT,
    use_class_weights=False,
    experiment_name="kfold_with_dropout",
    selected_folds=[3]
)

df_dropout = analyze_and_save_kfold_results(
    dropout_results,
    experiment_name="Dropout Mechanism",
    output_filename='output/results/kfold_dropout_results.csv'
)

### 7. L2 Regularization

In [ ]:
import copy

CONFIG_L2 = copy.deepcopy(CONFIG)
CONFIG_L2['l2_reg_lambda'] = 0.001
CONFIG_L2['num_epochs'] = 2

l2_reg_results = run_kfold_training(
    dataset=train_dataset,
    kfold=kfold,
    config=CONFIG_L2,
    use_class_weights=False,
    experiment_name="kfold_with_l2_reg",
    selected_folds=[3]
)

df_l2_reg = analyze_and_save_kfold_results(
    l2_reg_results,
    experiment_name="L2 Regularization",
    output_filename='output/results/kfold_l2_reg_results.csv'
)